In [ ]:
import sys
sys.path.append('/data/users/quentin/final_package/')
import scClone2DR as sccdr
import matplotlib.pyplot as plt
import numpy as np
from copy import deepcopy
import pickle
import torch
from tqdm import tqdm
import os
import plotly.io as pio
rootpath = '/data/users/quentin/final_package/experiments/paper_results'

COHORT = 'melanoma'
gene_set_collections = ['gene','c6','hallmarks', 'c2_pid', 'c2_kegg_medicus']
clonemodes = ['scatrex', 'phenograph']
gene_set_collection = gene_set_collections[3]
clonemode = clonemodes[0]
mode_features = 'metacells_{0}_{1}'.format(gene_set_collection, clonemode)

directory = os.path.join(rootpath,COHORT)
if not os.path.exists(directory):
    os.makedirs(directory)
pathsave = os.path.join(rootpath,COHORT,gene_set_collection)
if not os.path.exists(pathsave):
    os.makedirs(pathsave)
    
if COHORT=='melanoma':
    path_rna = '/data/users/04_share_reanalysis_results/melanoma_2025/02_atypical_removed_preprocessing/{0}/'.format(mode_features)
    path_fastdrug = '/data/users/04_share_reanalysis_results/melanoma_2025/MEL_CNN_abundances_no_plate_effect_correction.csv'
    concentration_DMSO = '100'
    concentration_drug = '5'
    path_info_cohort = None
elif COHORT=='aml':
    concentration_DMSO = '200'
    concentration_drug = '10'
    path_rna = '/data/users/04_share_reanalysis_results/01_aml/02_atypical_removed_preprocessing/{0}/'.format(mode_features)
    path_fastdrug = '/data/users/04_share_reanalysis_results/01_aml/AML_PCY_cell_numbers_no_plate_effect_correction.csv'
    path_info_cohort = '/data/users/04_share_reanalysis_results/01_aml/2024-08-15_aml_overview_scRNA.tsv'
model = sccdr.models.scClone2DR(path_fastdrug=path_fastdrug, path_rna=path_rna, path_info_cohort=path_info_cohort, type_guide='lowrank_MVN')
data_ref = model.get_real_data(concentration_DMSO=concentration_DMSO, concentration_drug=concentration_drug)

In [ ]:
penalties_l1 = [0.2*2**i for i in range(10)]
penalties_l2 = [0.05*2**i for i in range(10)]
print(penalties_l1, penalties_l2)

pl1, pl2 = 6.4, 3.2

folder_save = 'hyperparam_optim'

# Evaluating the stability of the learned parameters across different experiments

In [ ]:
def get_std_mean(pl1, pl2, param_studied='beta'):
    found = False
    ite = 0
    while (not(found) and ite<10):
        try:
            with open(os.path.join(rootpath,'{0}/{1}_{2}/{6}/params_svi_ite_{3}_l1_{4}_l2_{5}.pkl'.format(COHORT, gene_set_collection, clonemode, ite, pl1, pl2, folder_save)), 'rb') as handle:
                params = pickle.load(handle)
                dim = params['beta'].shape[1]
            found = True
            
        except:
            ite += 1
            pass
    if found:

        if param_studied == 'beta':
            beta_mean = np.zeros((47,dim))
            beta2_mean = np.zeros((47,dim))
        else:
            D = params['beta'].shape[0]
            N, Kmax = params['proportions'].shape
            beta_mean = np.zeros((D, Kmax, 30))
            beta2_mean = np.zeros((D, Kmax, 30))

        nbites = 25
        count = 0
        for ite in range(nbites):
            try:
                with open(os.path.join(rootpath,'{0}/{1}_{2}/{6}/params_svi_ite_{3}_l1_{4}_l2_{5}.pkl'.format(COHORT, gene_set_collection, clonemode, ite, pl1, pl2, folder_save)), 'rb') as handle:
                    params = pickle.load(handle)
                    if param_studied == 'beta':
                        beta_mean += params['beta']  /nbites
                        beta2_mean += params['beta']**2 /nbites
                    else:
                        samples_names_train = params['sample_names_train']
                        idxs_train = []
                        idxs_test = []
                        for sample in samples_names_train:
                            idx_s = np.where(model.sample_names==sample)[0][0]
                            idxs_train.append(idx_s)
                        params['proportions'] = params['proportions'][:len(idxs_train),:]
                        params['theta_fd'] = params['theta_fd'][:len(idxs_train)]
                        data_train, data_test, sample_names_train, sample_names_test = model.get_real_data_split(idxs_train, [])
                        postmeans = model.get_posterior_mean_latentvar(params, nsamples=100)
                        params.update(postmeans)
                        dd = model.merge_data_param(data_train, {key:val for key, val in params.items() if not('sample' in key)})
                        pi = model.get_survival_probas(dd)
                        pi = np.array(pi)
                        beta_mean += pi  /nbites
                        beta2_mean += pi**2 /nbites
                    count+=1
            except:
                pass
        beta_mean *= nbites/count
        beta2_mean *= nbites/count

        beta_std = beta2_mean - beta_mean**2
        return found, beta_std, beta_mean
    else:
        return found, [], []

found, beta_std, beta_mean = get_std_mean(pl1, pl2)

In [ ]:
found

In [ ]:
try:
    plt.figure(figsize=(20,3))
    b = plt.imshow(np.log10(np.abs(beta_std/beta_mean)))
    plt.colorbar(b)
except:
    pass

In [ ]:
plt.plot(beta_mean[1,:])
plt.xlabel('Genes', fontsize=14)
plt.title('$\\beta$ coefficients for the drug: '+model.FD.selected_drugs[10])
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def show_std_vs_mean(beta_std , beta_mean, param_studied='beta', savefig=False, color='navy'):
    
    mode = 'sorted_by_mean'
    
    cv_values = np.abs(beta_std / beta_mean).reshape(-1) 
        
    if mode == 'sorted_by_mean':
        idxs = np.argsort(np.abs(beta_mean).reshape(-1) )  # Sorted for visualization
        absc = np.abs(beta_mean).reshape(-1)[idxs]
    else:
        idxs = np.argsort(cv_sorted)
        # Create x-axis (percentile)
        absc = np.linspace(0, 100, len(cv_sorted))
    cv_sorted = cv_values[idxs]
    
    # Ensure `color` is a list of valid colors (this should already be passed in the correct format)
    # If color is a single string, convert it into a list of that color repeated
    if isinstance(color, str):
        color = [color] * len(cv_sorted)
    else:
        color = color[idxs]

    # Convert to log scale (handling zeros)
    cv_sorted = np.log10(cv_sorted + 1e-10)  # Adding small value to avoid log(0)



    # Plot with improved aesthetics
    plt.figure(figsize=(7,5))
    
    # Plotting the entire line using `color` passed as list
    plt.scatter(absc, cv_sorted, c=color, linewidth=2, s=1)  # Use single plot if colors match the data size

    # Labels and aesthetics

    try:
        # Find threshold index (e.g., log10(0.2))
        if param_studied == 'beta':
            plt.xlabel('Sorted Drug-Gene Pairs (%)', fontsize=12)
            plt.ylabel('$\log_{10}$(|std/mean|) on $\\beta_{dj}$', fontsize=12)


            threshold = np.log10(0.2)
            lab = '$\log_{10}(0.2)$'
        else:
            plt.xlabel('Sorted Sample-Drug-Subclone Triplets (%)', fontsize=12)
            plt.ylabel('$\log_{10}$(|std/mean|) on $\\pi_{idk}$', fontsize=12)

            threshold = np.log10(0.05)
            lab = '$\log_{10}(0.05)$'

        if mode != 'sorted_by_mean':
            idx = np.where(cv_sorted >= threshold)[0][0]  # First index above threshold
            plt.scatter([idx * 100 / len(cv_sorted)], [cv_sorted[idx]], 
                        color='red', marker='x', s=100, label=('Threshold: '+lab))
            plt.axhline(y=threshold, color='gray', linestyle='--', linewidth=1, alpha=0.7)  # Threshold line
            plt.axvline(x=idx * 100 / len(cv_sorted), color='red', linestyle='--', linewidth=1, alpha=0.7)  # Vertical threshold marker

            # Coordinates of the red point
            red_x = idx * 100 / len(cv_sorted)
            red_y = cv_sorted[idx]

            # Annotation of the red point
            plt.annotate(f'({red_x:.2f}%, '+lab+')', 
                         xy=(red_x, red_y), 
                         xytext=(red_x - 40, red_y + 0.4),  # Offset for better visibility
                         arrowprops=dict(arrowstyle="->", color='red'),
                         fontsize=12, color='red')
        else:
            if param_studied == 'beta':
                plt.xlabel('Mean of $\\beta_{dj}$ of Drug-Pathway Pairs $(d,j)$', fontsize=12)
            else:
                plt.xlabel('Mean of $\\pi_{idk}$ of Sample-Drug-Subclone Triplets $(i,d,k)$', fontsize=12)

            plt.axhline(y=threshold, color='red', linestyle='--', linewidth=1, alpha=0.9)  # Threshold line
            
            # Coordinates of the red point
            idx = int(0.999*len(absc))
            red_x = absc[idx]
            red_y = threshold*0.9

            # Annotation of the red point
            plt.annotate(lab, 
                         xy=(red_x, red_y), 
                         xytext=(red_x , red_y),  # Offset for better visibility
                         fontsize=12, color='red')
    except:
        pass

    plt.grid(True, linestyle='--', alpha=0.5)
    
    if savefig:
        plt.savefig(f'stability_analysis_{param_studied}.png', bbox_inches='tight')

    # Show plot
    plt.show()
    
def get_colors(n, cmap_name='viridis'):
    """Generate a list of `n` colors from a given matplotlib colormap."""
    cmap = plt.get_cmap(cmap_name)
    if True:
        return [cmap(i / (n - 1)) for i in range(n)]  # Get n evenly spaced colors
    else:
        colors = []
        mr = np.array(torch.sum(data_ref['masks']['R'],dim=0).float())
        for d in range(mr.shape[0]):
            colors.append(cmap(np.mean(mr[d,:]>0)))
        return colors

import matplotlib
# Example usage
if True:
    n_colors = beta_mean.shape[1]  # Ensure this matches the number of data points
    colors = get_colors(n_colors, "viridis")  # Choose colormap (e.g., "viridis")
    colors = [matplotlib.colors.to_hex(col) for col in colors]

    # Ensure colors is in the correct shape (if necessary, repeat colors across columns)
    colors = np.array(colors)
    colors = np.tile(colors, beta_mean.shape[0])

    # Flatten color array into a list of tuples, this might not be needed if you're already handling colors properly
    colors = colors.reshape(-1)

else:
    n_colors = beta_mean.shape[0]  # Ensure this matches the number of data points
    colors = get_colors(n_colors, "viridis")  # Choose colormap (e.g., "viridis")
    colors = [matplotlib.colors.to_hex(col) for col in colors]

    # Ensure colors is in the correct shape (if necessary, repeat colors across columns)
    colors = np.array(colors)
    colors = np.repeat(colors, beta_mean.shape[1], axis=0)

    # Flatten color array into a list of tuples, this might not be needed if you're already handling colors properly
    colors = colors.reshape(-1)

# Call the function
found, beta_std, beta_mean = get_std_mean(pl1, pl2)
show_std_vs_mean(beta_std, beta_mean, savefig=True, param_studied='beta', color='navy')


In [ ]:
found, beta_std, beta_mean = get_std_mean(pl1, pl2, param_studied='pi')
show_std_vs_mean(beta_std , beta_mean, param_studied='pi', savefig=True)

In [ ]:
for pl1 in penalties_l1:
    for pl2 in penalties_l2:
        print(pl1, pl2)
        found, beta_std, beta_mean = get_std_mean(pl1, pl2, param_studied='pi')
        if found:
            show_std_vs_mean(beta_std , beta_mean, param_studied='pi')

In [ ]:
beta_mean.shape

In [ ]:
for pl1 in penalties_l1:
    for pl2 in penalties_l2:
        print(pl1, pl2)
        found, beta_std, beta_mean = get_std_mean(pl1, pl2, param_studied='beta')
        if found:
            plt.plot(beta_mean[1,:])
            plt.xlabel('Genes', fontsize=14)
            plt.title('$\\beta$ coefficients for the drug: '+model.FD.selected_drugs[10])
            plt.show()

## Stability Index: Jaccard Similarity or Intersection Over Union (IoU)


In [ ]:
from sklearn.metrics import jaccard_score
support_sets = []

for ite in range(nbites):
    with open(os.path.join(rootpath,'{0}/{1}_{2}/stability/params_svi_ite_{3}_l1_{4}_l2_{5}.pkl'.format(COHORT, gene_set_collection, clonemode, ite, pl1, pl2)), 'rb') as handle:
        params = pickle.load(handle)
        selected_features = np.abs(params['beta']) > 1e-3  # Selected features (non-zero)
        support_sets.append(selected_features.flatten())

# Compute pairwise Jaccard similarity
jaccard_similarities = []
for i in range(nbites):
    for j in range(i + 1, nbites):
        jaccard_similarities.append(jaccard_score(support_sets[i], support_sets[j]))

avg_jaccard_index = np.mean(jaccard_similarities)
print(f'Average Jaccard Index: {avg_jaccard_index}')
std_jaccard_index = np.std(jaccard_similarities)
print(f'Std Jaccard Index: {std_jaccard_index}')

In [ ]:
plt.plot(np.sort(np.log10(np.abs(params['beta'].reshape(-1)))))

In [ ]:
np.mean(np.array(support_sets), axis=1)

In [ ]:
model.type_guide = "lowrank_MVN"

In [ ]:
rootpath = '/data/users/quentin/final_package/experiments/paper_results'
penalties_l1 = [0.2*2**i for i in range(10)]
penalties_l2 = [0.05*2**i for i in range(10)]
log_likeli = np.zeros((len(penalties_l1),len(penalties_l2)))
variances = np.zeros((len(penalties_l1),len(penalties_l2)))

MODE = 'TEST'
for pl1, penalty_l1 in tqdm(enumerate(penalties_l1)):
    for pl2, penalty_l2 in enumerate(penalties_l2):
        beta2_mean = np.zeros((47,500))
        beta_mean = np.zeros((47,500))
        count = 0
        log_likeli_temp = 0
        for ite in range(5):
            try:
                with open(os.path.join(rootpath,'{0}/{1}_{2}/stability/params_svi_ite_{5}_l1_{3}_l2_{4}.pkl'.format(COHORT, gene_set_collection, clonemode, str(penalty_l1), str(penalty_l2), ite)), 'rb') as handle:
                    params = pickle.load(handle)
                    beta2_mean += params['beta']**2
                    beta_mean += params['beta']
                    count += 1
                samples_names_train = params['sample_names_train']
                samples_names_test = params['sample_names_test']

                idxs_train = []
                idxs_test = []
                for sample in samples_names_train:
                    idx_s = np.where(model.sample_names==sample)[0][0]
                    idxs_train.append(idx_s)
                for sample in samples_names_test:
                    idx_s = np.where(model.sample_names==sample)[0][0]
                    idxs_test.append(idx_s)
                if MODE=='TEST':
                    params['proportions'] = params['proportions'][len(idxs_train):,:]
                    params['theta_fd'] = params['theta_fd'][len(idxs_train):]
                    data_train, data_test, sample_names_train, sample_names_test = model.get_real_data_split(idxs_train, idxs_test)
                    postmeans = model.get_posterior_mean_latentvar(params, nsamples=100)
                    params.update(postmeans)
                    log_likeli_temp += model.get_log_likelihood(data_test, {key:val for key,val in params.items() if not('sample_' in key)})
                else:
                    params['proportions'] = params['proportions'][:len(idxs_train),:]
                    params['theta_fd'] = params['theta_fd'][:len(idxs_train)]
                    data_train, data_test, sample_names_train, sample_names_test = model.get_real_data_split(idxs_train, idxs_test)
                    postmeans = model.get_posterior_mean_latentvar(params, nsamples=100)
                    params.update(postmeans)
                    log_likeli_temp += model.get_log_likelihood(data_train, {key:val for key,val in params.items() if not('sample_' in key)})
            
            except:
                print('error', pl1, pl2)
                pass
        beta2_mean /= count
        beta_mean /= count
        var = beta2_mean - beta_mean**2
        variances[pl1, pl2] = np.mean(np.sort(var.reshape(-1))[-100:])#np.mean(beta)
        log_likeli[pl1, pl2] = log_likeli_temp/count

In [ ]:
data_train['frac_r'].shape

In [ ]:
m = 4
mp = log_likeli.shape[0]-m
b = plt.imshow(log_likeli[m:,:][:,m:])
plt.colorbar(b)
plt.ylabel('L1 parameter')
plt.yticks(range(len(penalties_l1))[:mp], penalties_l1[m:])
plt.xticks(range(len(penalties_l2))[:mp], penalties_l2[m:])
plt.xlabel('L2 parameter')

In [ ]:
b = plt.imshow(variances)
plt.colorbar(b)
plt.ylabel('L1 parameter')
plt.yticks(range(len(penalties_l1)), penalties_l1)
plt.xticks(range(len(penalties_l2)), penalties_l2)
plt.xlabel('L2 parameter')

In [ ]:
print(penalties_l1, penalties_l2)

In [ ]:
ite = 4
with open(os.path.join(rootpath,'{0}/{1}_{2}/stability/params_svi_ite_{5}_l1_{3}_l2_{4}.pkl'.format(COHORT, gene_set_collection, clonemode, str(pl1), str(pl2), ite)), 'rb') as handle:
    params = pickle.load(handle)
import seaborn as sns
sns.clustermap(params['beta'][:,:])

In [ ]:
MODE = 'TEST'
for pl1, penalty_l1 in tqdm(enumerate(penalties_l1)):
    for pl2, penalty_l2 in enumerate(penalties_l2):
        beta = None
        count = 0
        log_likeli_temp = 0
        print(penalty_l1, penalty_l2)
        for ite in range(1):
            try:
                with open(os.path.join(rootpath,'{0}/{1}_{2}/{6}/params_svi_ite_{5}_l1_{3}_l2_{4}.pkl'.format(COHORT, gene_set_collection, clonemode, str(penalty_l1), str(penalty_l2), ite, folder_save)), 'rb') as handle:
                    params = pickle.load(handle)
                    if beta is None:
                        beta = (params['beta']**2-params['beta'])
                    else:
                        beta += (params['beta']**2-params['beta'])
                    count += 1
                    
                samples_names_train = params['sample_names_train']

                idxs_train = []
                for sample in samples_names_train:
                    idx_s = np.where(model.sample_names==sample)[0][0]
                    idxs_train.append(idx_s)
                if MODE=='TEST':
                    samples_names_test = params['sample_names_test']
                    idxs_test = []
                    for sample in samples_names_test:
                        idx_s = np.where(model.sample_names==sample)[0][0]
                        idxs_test.append(idx_s)
                    params['proportions'] = params['proportions'][len(idxs_train):,:]
                    params['theta_fd'] = params['theta_fd'][len(idxs_train):]
                    data_train, data_test, sample_names_train, sample_names_test = model.get_real_data_split(idxs_train, idxs_test)
                    postmeans = model.get_posterior_mean_latentvar(params, nsamples=100)
                    params.update(postmeans)
                    res = model.sampling(data_test, {key:val for key,val in params.items() if not('sample_' in key)})
                    sccdr.resultanalysis.scatter_counts(data_test, res, color_mode='patient')
                    plt.show()
                else:
                    params['proportions'] = params['proportions'][:len(idxs_train),:]
                    params['theta_fd'] = params['theta_fd'][:len(idxs_train)]
                    data_train, data_test, sample_names_train, sample_names_test = model.get_real_data_split(idxs_train, [])
                    postmeans = model.get_posterior_mean_latentvar(params, nsamples=100)
                    params.update(postmeans)
                    res = model.sampling(data_train, {key:val for key,val in params.items() if not('sample_' in key)})
                    sccdr.resultanalysis.scatter_counts(data_train, res, color_mode='patient')
                    plt.show()
            except:
                print('error', pl1, pl2)
                pass



In [ ]:
mode_features = 'metacells_{0}_{1}'.format(gene_set_collection, clonemode)

with open(os.path.join(rootpath,'{0}/{1}_{2}/params_svi.pkl'.format(COHORT, gene_set_collection, clonemode)), 'rb') as handle:
    params_svi = pickle.load(handle)

import h5py
with h5py.File(os.path.join(rootpath, COHORT, gene_set_collection+'_'+clonemode, 'local_importance.h5'), "r") as f:
    # Print all root level object names (aka keys) 
    # these can be group or dataset names 
    print("Keys: %s" % f.keys())
    # get first object name/key; may or may NOT be a group
    print(list(f['local_importance_mean'].shape))
    print(f['local_importance_mean'].attrs.keys())
    drugs = f['local_importance_mean'].attrs['dim1_drugs']
    dims = f['local_importance_mean'].attrs['dim4_dimensions']


filepath = os.path.join(rootpath,'{0}/{1}_{2}/beta.h5'.format(COHORT, gene_set_collection, clonemode))
with h5py.File(filepath, 'w') as f:
    # Create a dataset
    dset = f.create_dataset('beta', data=params_svi['beta'])

    column_names = {
    'dim2_dimensions': dims,
    'dim1_drugs': drugs
    }

    for dim, labels in column_names.items():
        dset.attrs[dim] = labels

In [ ]:
dims

In [ ]:
mode_features = 'metacells_{0}_{1}'.format(gene_set_collection, clonemode)

penall1 = 12.8
penall2 = 3.2
with open(os.path.join(rootpath,'{0}/{1}_{2}/stability/params_svi_ite_0_l1_{3}_l2_{4}.pkl'.format(COHORT, gene_set_collection, clonemode, str(penall1), str(penall2))), 'rb') as handle:
    params_svi = pickle.load(handle)

import h5py
with h5py.File(os.path.join(rootpath, COHORT, gene_set_collection, 'local_importance.h5'), "r") as f:
    # Print all root level object names (aka keys) 
    # these can be group or dataset names 
    print("Keys: %s" % f.keys())
    # get first object name/key; may or may NOT be a group
    print(list(f['local_importance_mean'].shape))
    print(f['local_importance_mean'].attrs.keys())
    drugs = f['local_importance_mean'].attrs['dim1_drugs']
    dims = f['local_importance_mean'].attrs['dim4_dimensions']


filepath = os.path.join(rootpath,'{0}/{1}/beta.h5'.format(COHORT, gene_set_collection))
with h5py.File(filepath, 'w') as f:
    # Create a dataset
    dset = f.create_dataset('beta', data=params_svi['beta'])

    column_names = {
    'dim2_dimensions': dims,
    'dim1_drugs': drugs
    }

    for dim, labels in column_names.items():
        dset.attrs[dim] = labels

In [ ]:
import seaborn as sns
sns.clustermap(params_svi['beta'][:,:])

In [ ]:
idxdrug = 20
plt.plot(params['beta'][idxdrug,:])
plt.xlabel('Genes', fontsize=14)
plt.title('$\\beta$ coefficients for the drug: '+model.FD.selected_drugs[idxdrug])
plt.show()

In [ ]:
x = np.abs(params['beta'][10,:])
x = np.sort(x)
plt.plot(x, np.cumsum(np.ones(len(x))))
plt.ylabel('Genes', fontsize=14)
plt.xlabel('$\\beta$ coefficients for the drug: '+model.FD.selected_drugs[10], fontsize=14)
plt.show()
print(np.mean(params['beta'][10,:]>+))

In [ ]:
x = np.abs(params['beta'].reshape(-1))
x = np.sort(x)
idx = np.where(x>=0.02)[0][0]
x = np.sort(x[idx:])
plt.plot(x, np.cumsum(np.ones(len(x))))
plt.ylabel('Genes', fontsize=14)
plt.xlabel('$\\beta$ coefficients for the drug: '+model.FD.selected_drugs[10], fontsize=14)
plt.show()
print(len(x)/len(params['beta'].reshape(-1)))

In [ ]:
with open(os.path.join(rootpath,'{0}/{1}_{2}/posterior_mean_params.pkl'.format(COHORT, gene_set_collection, clonemode)), 'rb') as handle:
    res = pickle.load(handle)

In [ ]:
sccdr.resultanalysis.show_fractions(data_ref, res, idxdrug=0)

In [ ]:
sccdr.resultanalysis.show_cells(data_ref, res)

In [ ]:
sccdr.resultanalysis.scatter_counts(data_ref, res, color_mode='patient')

In [ ]:
for idxdrug in range(20):
    if COHORT=='melanoma':
        sccdr.resultanalysis.survival_probabilities(data_ref, res['pi'].detach().numpy(), model.cluster2clonelabel, idxdrug=idxdrug)
    else:
        sccdr.resultanalysis.survival_probabilities(data_ref, res['pi'].detach().numpy(), model.cluster2clonelabel, df_info_cohort=model.dfinfo, idxdrug=idxdrug)

In [ ]:
for sampleID in range(model.N):
    sccdr.resultanalysis.survival_probabilities_relative_by_patient_optimized(data_ref, res['pi'].detach().numpy(), params_svi['proportions'], model.cluster2clonelabel, model.cluster2cat, model.cat2clusters, model.FD.selected_drugs, sampleID=sampleID, savefig='cluster_drug_sample_4.png')

In [ ]:
# Stratifying by samples
pi = np.nan_to_num(deepcopy(res['pi'].detach().numpy()))
ratio_pi = deepcopy(pi)
for idxdrug in range(pi.shape[0]):
    for sampleid in range(pi.shape[2]):
        learned_props = params_svi['proportions'][sampleid,:]
        healthy_survival =  np.sum((pi[idxdrug,:,sampleid]*learned_props)[model.cat2clusters['healthy']])
        healthy_survival /=  np.sum(learned_props[model.cat2clusters['healthy']])
        ratio_pi[idxdrug,:,sampleid] /= healthy_survival

for i in range(1,3):
    sccdr.resultanalysis.survival_probabilities_relative_by_patient(data_ref, ratio_pi, res['pi'].detach().numpy(), model.cluster2clonelabel, model.cluster2cat, model.FD.selected_drugs, sampleID=i)

In [ ]:
fold_changes_pred = []
fold_changes_notpred = []

learned_props = np.zeros((len(model.sample_names), model.Kmax))

res = get_res_from_sample(model, params_svi, data_ref)
ls_vars = ['n0_c', 'n0_r', 'n_r', 'n_c', 'n_rna', 'pi', 'nu_healthy_drug', 'nu_healthy_control']
for var in ls_vars:
    res[var] = torch.zeros(res[var].shape)


for id_patient in tqdm(range(len(model.sample_names))):
    patient = model.sample_names[id_patient]

    idxs_train = [i for i in range(int(data_ref['N'])) if i!=id_patient]
    idxs_test = [id_patient]
    data_train, data_test, sample_names_train, sample_names_test = model.get_real_data_split(idxs_train, idxs_test)
    params_svi_validation = {}
    for key, val in params_svi.items():
        if torch.is_tensor(val):
            params_svi_validation[key] = val.clone().detach()
        else:
            params_svi_validation[key] = val
    params_svi_validation['proportions'] = params_svi_validation['proportions'][[len(idxs_train)],:]
    params_svi_validation['theta_fd'] = params_svi_validation['theta_fd'][[len(idxs_train)]]
    data_validation = deepcopy(data_test)

    learned_props[id_patient,:] = params_svi['proportions'][len(idxs_train),:]

    if True:
        nb_ites = 100
        for ite in range(nb_ites):
            restmp = get_res_from_sample(model, params_svi_validation, data_validation)
            res['n0_c'][:,id_patient] += restmp['n0_c'][:,0]/nb_ites
            res['n_c'][:,id_patient] += restmp['n_c'][:,0]/nb_ites
            res['n0_r'][:,:,id_patient] += restmp['n0_r'][:,:,0]/nb_ites
            res['n_rna'][:,id_patient] += restmp['n_rna'][:,0]/nb_ites
            res['pi'][:,:,id_patient] += restmp['pi'][:,:,0]/nb_ites
            res['nu_healthy_drug'][:,:,id_patient] += restmp['nu_healthy_drug'][:,:,0]/nb_ites
            res['nu_healthy_control'][:,id_patient] += restmp['nu_healthy_control'][:,0]/nb_ites
    else:
        nb_ites = 100
        from torch.distributions import MultivariateNormal
        loc = params_svi_validation['AutoMultivariateNormal.loc']
        
        # Define the scale_tril (lower triangular matrix)
        scale_tril = params_svi_validation['AutoMultivariateNormal.scale_tril']
        
        # Define the multivariate Gaussian distribution
        mvn = MultivariateNormal(loc=torch.tensor(loc), scale_tril=torch.tensor(scale_tril))
        gamma = mvn.sample()
        dim = int(len(gamma)/model.n_clonelabels)
        for i, clonelabel in enumerate(model.clonelabels):
            params_svi_validation['gamma_{0}'.format(clonelabel)] = torch.zeros(gamma[i*dim:(i+1)*dim].shape)
        for ite in range(nb_ites):
            gamma = mvn.sample()
            dim = int(len(gamma)/model.n_clonelabels)
            for i, clonelabel in enumerate(model.clonelabels):
                params_svi_validation['gamma_{0}'.format(clonelabel)] += gamma[i*dim:(i+1)*dim]/nb_ites
                
        restmp = get_res_from_sample(model, params_svi_validation, data_validation)
        res['n0_c'][:,id_patient] = restmp['n0_c'][:,0]
        res['n_c'][:,id_patient] = restmp['n_c'][:,0]
        res['n0_r'][:,:,id_patient] = restmp['n0_r'][:,:,0]
        res['n_rna'][:,id_patient] = torch.tensor(restmp['n_rna'][:,0])
        res['pi'][:,:,id_patient] = restmp['pi'][:,:,0]
        res['nu_healthy_drug'][:,:,id_patient] = restmp['nu_healthy_drug'][:,:,0]
        res['nu_healthy_control'][:,id_patient] = restmp['nu_healthy_control'][:,0]

    params_fold_change = {'pi': res['pi'][:,:,[id_patient]],
                          'nu_healthy_drug': res['nu_healthy_drug'][:,:,[id_patient]],
                          'nu_healthy_control': res['nu_healthy_control'][:,[id_patient]],
                          'proportions': params_svi_validation['proportions']
                        }
    fold_changes, pi, colors = model.get_fold_change(data_validation, params_fold_change, output_results=True)
    fold_changes_pred += list(fold_changes['pred'])
    fold_changes_notpred += list(fold_changes['not pred'])

    # fold_changes, pi, colors = model.get_fold_change(data_validation, params_svi_validation, output_results=True)
    # fold_changes_pred_test.append(fold_changes['pred'])
    # fold_changes_notpred_test.append(fold_changes['not pred'])